In [1]:
import numpy as np
import nibabel as nib
import trimesh

from scipy.ndimage import binary_closing, binary_fill_holes, distance_transform_edt
from skimage.measure import marching_cubes


# ============================================================
# SETTINGS
# ============================================================

CLOSING_RADIUS = 2

LABEL_FILE = "Brain_label.nii"
OUTPUT_OBJ_COLOR = f"brain_segmented_colored_{CLOSING_RADIUS:02d}.obj"
OUTPUT_OBJ_GREY = f"brain_segmented_grey_{CLOSING_RADIUS:02d}.obj"
OUTPUT_COLOR_CSV = f"brain_color_{CLOSING_RADIUS:02d}.csv"

MIN_LABEL = 1
MAX_LABEL = 86

# Keep your existing COLORS dictionary here.
# COLORS = {1: (...), ..., 86: (...)}

FALLBACK_COLOR = (128, 128, 128, 255)
SAMPLE_OFFSET = 0.75

# ============================================================
# ANATOMICAL COLORS
# ============================================================

COLORS = {
    1:  (237, 216, 166),
    2:  (213, 91, 43),
    3:  (213, 91, 43),
    4:  (244, 201, 68),
    5:  (244, 201, 68),
    6:  (178, 210, 67),
    7:  (178, 210, 67),
    8:  (108, 186, 57),
    9:  (108, 186, 57),
    10: (186, 235, 189),
    11: (186, 235, 189),
    12: (83, 149, 186),
    13: (83, 149, 186),
    14: (56, 70, 132),
    15: (56, 70, 132),
    16: (172, 217, 111),
    17: (172, 217, 111),
    18: (209, 134, 193),
    19: (209, 134, 193),
    20: (203, 74, 150),
    21: (203, 74, 150),
    22: (174, 117, 72),
    23: (174, 117, 72),
    24: (243, 228, 80),
    25: (243, 228, 80),
    26: (241, 250, 215),
    27: (241, 250, 215),
    28: (126, 199, 89),
    29: (126, 199, 89),
    30: (187, 150, 147),
    31: (187, 150, 147),
    32: (248, 213, 200),
    33: (248, 213, 200),
    34: (201, 60, 45),
    35: (201, 60, 45),
    36: (140, 196, 59),
    37: (140, 196, 59),
    38: (204, 58, 50),
    39: (204, 58, 50),
    40: (192, 235, 230),
    41: (192, 235, 230),
    42: (168, 221, 229),
    43: (168, 221, 229),
    44: (60, 58, 153),
    45: (60, 58, 153),
    46: (113, 75, 174),
    47: (113, 75, 174),
    48: (245, 220, 74),
    49: (245, 220, 74),
    50: (123, 191, 57),
    51: (123, 191, 57),
    52: (233, 134, 173),
    53: (233, 134, 173),
    54: (229, 128, 61),
    55: (229, 128, 61),
    56: (246, 224, 100),
    57: (246, 224, 100),
    58: (90, 132, 53),
    59: (90, 132, 53),
    60: (191, 220, 96),
    61: (191, 220, 96),
    62: (237, 242, 251),
    63: (237, 242, 251),
    64: (203, 226, 237),
    65: (203, 226, 237),
    66: (146, 133, 204),
    67: (146, 133, 204),
    68: (185, 231, 228),
    69: (185, 231, 228),
    70: (249, 237, 139),
    71: (247, 205, 84),
    72: (148, 65, 176),
    73: (251, 245, 250),
    74: (119, 38, 29),
    75: (150, 198, 65),
    76: (219, 95, 74),
    77: (255, 255, 255),
    78: (213, 115, 44),
    79: (213, 115, 44),
    80: (176, 213, 233),
    81: (134, 88, 185),
    82: (134, 88, 185),
    83: (206, 50, 70),
    84: (206, 50, 70),
    85: (134, 88, 185),
    86: (134, 88, 185),
}


# ============================================================
# HELPERS
# ============================================================

def sample_nearest(volume, points, default=0):
    """Sample a 3-D volume at voxel-space points using nearest neighbours."""
    coordinates = np.rint(points).astype(np.intp)
    result = np.full(len(points), default, dtype=volume.dtype)

    valid = np.all(
        (coordinates >= 0)
        & (coordinates < np.asarray(volume.shape)),
        axis=1,
    )

    valid_coordinates = coordinates[valid]
    result[valid] = volume[
        valid_coordinates[:, 0],
        valid_coordinates[:, 1],
        valid_coordinates[:, 2],
    ]

    return result


def normalize_vectors(vectors):
    lengths = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / np.maximum(lengths, 1e-12)


# ============================================================
# LOAD LABEL VOLUME
# ============================================================

nii = nib.load(LABEL_FILE)

# Labels are discrete; explicitly convert them back to integers.
labels = np.rint(nii.get_fdata()).astype(np.int16)

if labels.ndim != 3:
    raise ValueError(f"Expected a 3-D label volume, got shape {labels.shape}")

print("Volume shape:", labels.shape)

# Keep the original anatomy separate from the cleaned mask.
# This lets closing/filling-created surface regions inherit the label
# (and therefore predefined color) of the nearest original anatomy.
anatomical_mask = (
    (labels >= MIN_LABEL)
    & (labels <= MAX_LABEL)
)

if not np.any(anatomical_mask):
    raise ValueError("No anatomical labels were found in the label volume.")

# For every voxel, store the coordinates of the closest original
# anatomical voxel. distance_transform_edt finds the nearest False voxel,
# so invert anatomical_mask here.
_, nearest_anatomical_indices = distance_transform_edt(
    ~anatomical_mask,
    return_indices=True,
)


# ============================================================
# CREATE AND CLEAN BRAIN MASK
# ============================================================

binary_mask = (labels >= MIN_LABEL) & (labels <= MAX_LABEL)

print("Brain voxels:", binary_mask.sum())

if CLOSING_RADIUS > 0:
    structure_size = 2 * CLOSING_RADIUS + 1
    structure = np.ones((structure_size,) * 3, dtype=bool)

    binary_mask = binary_closing(binary_mask, structure=structure)

    print("Brain voxels after closing:", binary_mask.sum())

binary_mask = binary_fill_holes(
    binary_mask,
    structure=np.ones((3, 3, 3), dtype=bool),
)

print("Brain voxels after filling enclosed holes:", binary_mask.sum())


# ============================================================
# MARCHING CUBES
# ============================================================

print("Running marching cubes...")

verts_voxel, faces, normals_voxel, _ = marching_cubes(
    binary_mask.astype(np.uint8),
    level=0.5,
)

print("Vertices:", len(verts_voxel))
print("Faces:", len(faces))

# Right after marching_cubes(...)
repeated_index = (
    (faces[:, 0] == faces[:, 1])
    | (faces[:, 1] == faces[:, 2])
    | (faces[:, 2] == faces[:, 0])
)

tri = verts_voxel[faces]
double_area = np.linalg.norm(
    np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0]),
    axis=1,
)
zero_area = double_area <= 1e-12

print("Faces with repeated indices:", repeated_index.sum())
print("Faces with zero area:", zero_area.sum())

keep = ~(repeated_index | zero_area)
faces = faces[keep]

print("Faces after cleanup:", len(faces))


# ============================================================
# CONVERT VERTICES AND NORMALS TO WORLD SPACE
# ============================================================

verts_world = nib.affines.apply_affine(nii.affine, verts_voxel)

# A normal transforms differently from a position:
# for row-vector normals, n_world = n_voxel @ inverse(linear_affine).
linear_affine = nii.affine[:3, :3]
normals_world = normals_voxel @ np.linalg.inv(linear_affine)
normals_world = normalize_vectors(normals_world)


# ============================================================
# ASSIGN ONE ANATOMICAL LABEL PER FACE
# ============================================================

# Face centres are needed because this mesh uses one predefined color
# per triangle.
face_centers_voxel = verts_voxel[faces].mean(axis=1)

# Derive a triangle direction by averaging the marching-cubes vertex
face_normals_voxel = normals_voxel[faces].mean(axis=1)
face_normals_voxel = normalize_vectors(face_normals_voxel)

side_a = face_centers_voxel + SAMPLE_OFFSET * face_normals_voxel
side_b = face_centers_voxel - SAMPLE_OFFSET * face_normals_voxel

inside_a = sample_nearest(binary_mask, side_a, default=False).astype(bool)
inside_b = sample_nearest(binary_mask, side_b, default=False).astype(bool)

# Sample from the side lying inside the cleaned brain mask.
sample_points = face_centers_voxel.copy()
sample_points[inside_a & ~inside_b] = side_a[inside_a & ~inside_b]
sample_points[inside_b & ~inside_a] = side_b[inside_b & ~inside_a]

# First attempt: obtain a label directly from the original label image.
face_labels = sample_nearest(labels, sample_points, default=0).astype(np.int16)

valid_face_labels = (
    (face_labels >= MIN_LABEL)
    & (face_labels <= MAX_LABEL)
)

# Closing and hole filling can create regions that were background in the
# original label image. Give those faces the label of the closest original
# anatomical voxel, so they use an existing predefined COLORS entry.
faces_needing_nearest_label = np.flatnonzero(~valid_face_labels)

if len(faces_needing_nearest_label) > 0:
    nearest_query_coords = np.rint(
        sample_points[faces_needing_nearest_label]
    ).astype(np.intp)

    # Clamp points at the volume boundary before indexing.
    nearest_query_coords = np.clip(
        nearest_query_coords,
        0,
        np.asarray(labels.shape) - 1,
    )

    nearest_source_coords = nearest_anatomical_indices[
        :,
        nearest_query_coords[:, 0],
        nearest_query_coords[:, 1],
        nearest_query_coords[:, 2],
    ]

    face_labels[faces_needing_nearest_label] = labels[
        tuple(nearest_source_coords)
    ]

# The large nearest-neighbour lookup is no longer needed.
del nearest_anatomical_indices

print("Faces using directly sampled labels:",
      len(faces) - len(faces_needing_nearest_label))
print("Faces assigned nearest anatomical labels:",
      len(faces_needing_nearest_label))
print("Faces still without a valid label:",
      (~valid_face_labels).sum())

# ============================================================
# CREATE FACE COLORS FROM PREDEFINED COLORS
# ============================================================

face_colors = np.zeros((len(faces), 4), dtype=np.uint8)

for label, rgb in COLORS.items():
    matching_faces = face_labels == label
    face_colors[matching_faces, :3] = rgb
    face_colors[matching_faces, 3] = 255

# Faces whose sampled label was outside 1–86 remain black.
unassigned_faces = face_colors[:, 3] == 0
face_colors[unassigned_faces] = (0, 0, 0, 255)

assert face_colors.shape == (len(faces), 4)

# ============================================================
# CREATE AND EXPORT MESHES
# ============================================================

mesh = trimesh.Trimesh(
    vertices=verts_world,
    faces=faces,
    vertex_normals=normals_world,
    process=False,
)

mesh.visual.face_colors = face_colors

mesh.export(OUTPUT_OBJ_COLOR, include_normals=True)

# Mesh 2 Grey

mesh = trimesh.Trimesh(
    vertices=verts_world,
    faces=faces,
    vertex_normals=normals_world,
    process=False,
)

mesh.export(OUTPUT_OBJ_GREY, include_normals=True)

# Color csv

face_color_table = np.column_stack((
    np.arange(len(faces), dtype=np.int64),
    face_labels.astype(np.int64),
    face_colors.astype(np.int64),
))
np.savetxt(
    OUTPUT_COLOR_CSV,
    face_color_table,
    delimiter=",",
    fmt="%d",
    header="face_index,label,red,green,blue,alpha",
    comments="",
)

print("\nSaved:")
print(" ", OUTPUT_OBJ_COLOR)
print("faces:", mesh.faces.shape)
print("vertices:", mesh.vertices.shape)

Volume shape: (149, 208, 140)
Brain voxels: 814476
Brain voxels after closing: 934914
Brain voxels after filling enclosed holes: 944982
Running marching cubes...
Vertices: 102728
Faces: 205472
Faces with repeated indices: 0
Faces with zero area: 0
Faces after cleanup: 205472
Faces using directly sampled labels: 162925
Faces assigned nearest anatomical labels: 42547
Faces still without a valid label: 42547

Saved:
  brain_segmented_colored_02.obj
faces: (205472, 3)
vertices: (102728, 3)
